# COT Data: Positioning & Sentiment Analysis

**CQF-Level Example**: Analyze CFTC Commitment of Traders data:
- Commercial vs Speculative positioning
- Net positions and extremes
- Positioning changes as predictive signals
- Cross-asset positioning analysis
- COT-based trading strategies

**Connectors Used:**
- `qj.cftc` - CFTC COT reports
- `qj.eod` - Asset prices for backtesting

**API:** https://api.quantjourney.cloud

## Run Output

![23_cot_positioning_sentiment](../plots/23_cot_positioning_sentiment_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "png"
from scipy import stats

import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Explore Available COT Symbols

In [ ]:
# Get popular COT symbols
popular = qj.cftc.get_popular_symbols()
popular_data = popular.get('value', popular) if isinstance(popular, dict) else popular

if isinstance(popular_data, list):
    print("Popular COT Symbols:")
    for item in popular_data[:15]:
        if isinstance(item, dict):
            print(f"  {item.get('symbol', item.get('name', item))}")
        else:
            print(f"  {item}")
else:
    print("Using default symbols...")
    popular_data = [
        {'symbol': 'ES', 'name': 'E-Mini S&P 500'},
        {'symbol': 'GC', 'name': 'Gold'},
        {'symbol': 'CL', 'name': 'Crude Oil'},
        {'symbol': 'ZN', 'name': '10-Year T-Note'},
        {'symbol': 'EC', 'name': 'Euro FX'},
        {'symbol': 'ZC', 'name': 'Corn'},
        {'symbol': 'ZW', 'name': 'Wheat'},
        {'symbol': 'NG', 'name': 'Natural Gas'}
    ]


## 2. Fetch COT Data for Key Assets

In [ ]:
# Fetch S&P 500 (ES) COT data
es_cot = qj.cftc.get_cot_data(symbol='ES')
es_data = es_cot.get('value', es_cot) if isinstance(es_cot, dict) else es_cot

if isinstance(es_data, list) and len(es_data) > 0:
    es_df = pd.DataFrame(es_data)
    es_df['date'] = pd.to_datetime(es_df.get('date', es_df.get('report_date')))
    es_df = es_df.set_index('date').sort_index()
    print(f"E-Mini S&P 500 COT: {len(es_df)} reports")
    print(es_df.head())
else:
    # Generate synthetic COT data
    print("Generating synthetic COT data for demo...")
    dates = pd.date_range(end=pd.Timestamp.now(), periods=260, freq='W-TUE')
    np.random.seed(42)
    
    es_df = pd.DataFrame({
        'commercial_long': 300000 + np.cumsum(np.random.randn(260) * 5000),
        'commercial_short': 350000 + np.cumsum(np.random.randn(260) * 5000),
        'noncommercial_long': 200000 + np.cumsum(np.random.randn(260) * 8000),
        'noncommercial_short': 180000 + np.cumsum(np.random.randn(260) * 8000),
        'open_interest': 2500000 + np.cumsum(np.random.randn(260) * 20000)
    }, index=dates)
    print(f"Synthetic COT data: {len(es_df)} reports")


In [ ]:
# Calculate key metrics
# Find the correct column names
long_cols = [c for c in es_df.columns if 'long' in c.lower() and 'noncommercial' not in c.lower() and 'nonreport' not in c.lower()]
short_cols = [c for c in es_df.columns if 'short' in c.lower() and 'noncommercial' not in c.lower() and 'nonreport' not in c.lower()]

# Commercial net position
if 'commercial_long' in es_df.columns:
    es_df['commercial_net'] = es_df['commercial_long'] - es_df['commercial_short']
    es_df['noncommercial_net'] = es_df['noncommercial_long'] - es_df['noncommercial_short']
else:
    # Try alternative column names
    print(f"Available columns: {es_df.columns.tolist()}")

# Open interest normalization
if 'open_interest' in es_df.columns:
    es_df['commercial_net_pct'] = es_df['commercial_net'] / es_df['open_interest'] * 100
    es_df['noncommercial_net_pct'] = es_df['noncommercial_net'] / es_df['open_interest'] * 100

# Z-scores for extreme positioning
lookback = 52  # 1 year
es_df['commercial_zscore'] = (
    (es_df['commercial_net'] - es_df['commercial_net'].rolling(lookback).mean()) /
    es_df['commercial_net'].rolling(lookback).std()
)
es_df['noncommercial_zscore'] = (
    (es_df['noncommercial_net'] - es_df['noncommercial_net'].rolling(lookback).mean()) /
    es_df['noncommercial_net'].rolling(lookback).std()
)

print("\nLatest COT Positioning:")
print(f"  Commercial Net: {es_df['commercial_net'].iloc[-1]:,.0f}")
print(f"  Commercial Z-Score: {es_df['commercial_zscore'].iloc[-1]:.2f}")
print(f"  Non-Commercial Net: {es_df['noncommercial_net'].iloc[-1]:,.0f}")
print(f"  Non-Commercial Z-Score: {es_df['noncommercial_zscore'].iloc[-1]:.2f}")


## 3. Positioning Visualization

In [ ]:
# Plot net positions
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=['Commercial Net Position', 'Non-Commercial (Speculator) Net', 'Z-Scores'],
    vertical_spacing=0.08
)

# Commercial
fig.add_trace(
    go.Scatter(
        x=es_df.index, y=es_df['commercial_net'],
        mode='lines', name='Commercial',
        fill='tozeroy', fillcolor='rgba(0,100,255,0.3)',
        line=dict(color='blue')
    ),
    row=1, col=1
)

# Non-Commercial (Speculators)
fig.add_trace(
    go.Scatter(
        x=es_df.index, y=es_df['noncommercial_net'],
        mode='lines', name='Speculators',
        fill='tozeroy', fillcolor='rgba(255,100,0,0.3)',
        line=dict(color='orange')
    ),
    row=2, col=1
)

# Z-Scores
fig.add_trace(
    go.Scatter(
        x=es_df.index, y=es_df['commercial_zscore'],
        mode='lines', name='Commercial Z',
        line=dict(color='blue')
    ),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(
        x=es_df.index, y=es_df['noncommercial_zscore'],
        mode='lines', name='Speculator Z',
        line=dict(color='orange')
    ),
    row=3, col=1
)
fig.add_hline(y=2, line_dash="dash", line_color="red", row=3, col=1)
fig.add_hline(y=-2, line_dash="dash", line_color="green", row=3, col=1)

fig.update_layout(
    title='E-Mini S&P 500 COT Positioning',
    template='plotly_dark',
    height=700
)
fig.show()


## 4. Multi-Asset COT Dashboard

In [ ]:
# Fetch COT for multiple assets
assets = {
    'ES': 'S&P 500',
    'GC': 'Gold',
    'CL': 'Crude Oil',
    'EC': 'EUR/USD',
    'ZN': '10Y Treasury'
}

cot_data = {}
for symbol, name in assets.items():
    try:
        response = qj.cftc.get_cot_data(symbol=symbol)
        data = response.get('value', response) if isinstance(response, dict) else response
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df.get('date', df.get('report_date')))
            df = df.set_index('date').sort_index()
            cot_data[symbol] = df
            print(f"✓ {name}: {len(df)} reports")
    except Exception as e:
        print(f"✗ {name}: {e}")

# Fallback to synthetic if needed
if len(cot_data) < 3:
    print("\nUsing synthetic multi-asset COT data...")
    dates = pd.date_range(end=pd.Timestamp.now(), periods=156, freq='W-TUE')
    np.random.seed(42)
    
    for symbol, name in assets.items():
        if symbol not in cot_data:
            cot_data[symbol] = pd.DataFrame({
                'commercial_net': np.cumsum(np.random.randn(156) * 5000),
                'noncommercial_net': np.cumsum(np.random.randn(156) * 8000),
                'open_interest': 500000 + np.cumsum(np.random.randn(156) * 10000)
            }, index=dates)


In [ ]:
# Calculate z-scores for all assets
zscore_summary = []

for symbol, df in cot_data.items():
    if 'commercial_net' not in df.columns:
        if 'commercial_long' in df.columns:
            df['commercial_net'] = df['commercial_long'] - df['commercial_short']
            df['noncommercial_net'] = df['noncommercial_long'] - df['noncommercial_short']
    
    lookback = min(52, len(df) - 1)
    
    if 'commercial_net' in df.columns:
        mean_comm = df['commercial_net'].rolling(lookback).mean().iloc[-1]
        std_comm = df['commercial_net'].rolling(lookback).std().iloc[-1]
        zscore_comm = (df['commercial_net'].iloc[-1] - mean_comm) / std_comm if std_comm > 0 else 0
        
        mean_spec = df['noncommercial_net'].rolling(lookback).mean().iloc[-1]
        std_spec = df['noncommercial_net'].rolling(lookback).std().iloc[-1]
        zscore_spec = (df['noncommercial_net'].iloc[-1] - mean_spec) / std_spec if std_spec > 0 else 0
        
        zscore_summary.append({
            'Symbol': symbol,
            'Name': assets[symbol],
            'Commercial Net': df['commercial_net'].iloc[-1],
            'Commercial Z': zscore_comm,
            'Speculator Net': df['noncommercial_net'].iloc[-1],
            'Speculator Z': zscore_spec
        })

zscore_df = pd.DataFrame(zscore_summary)
print("\nCOT Z-Score Summary (52-week lookback):")
print(zscore_df.to_string(index=False))


In [ ]:
# Z-Score heatmap
heatmap_data = zscore_df.set_index('Symbol')[['Commercial Z', 'Speculator Z']]

fig = px.imshow(
    heatmap_data,
    labels=dict(color="Z-Score"),
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    zmin=-3, zmax=3,
    aspect='auto'
)

# Add annotations
for i, symbol in enumerate(heatmap_data.index):
    for j, col in enumerate(heatmap_data.columns):
        val = heatmap_data.loc[symbol, col]
        fig.add_annotation(
            x=j, y=i,
            text=f"{val:.2f}",
            showarrow=False,
            font=dict(color='white' if abs(val) > 1 else 'black', size=14)
        )

fig.update_layout(
    title='Multi-Asset COT Z-Scores (Extremes at ±2)',
    template='plotly_dark',
    height=400
)
fig.show()


## 5. Positioning Change Analysis

In [ ]:
# Weekly change in positioning
es_df['commercial_change'] = es_df['commercial_net'].diff()
es_df['noncommercial_change'] = es_df['noncommercial_net'].diff()

# 4-week cumulative change
es_df['commercial_4w_change'] = es_df['commercial_net'].diff(4)
es_df['noncommercial_4w_change'] = es_df['noncommercial_net'].diff(4)

print("\nRecent Position Changes:")
print(f"  Commercial 1W Change: {es_df['commercial_change'].iloc[-1]:+,.0f}")
print(f"  Commercial 4W Change: {es_df['commercial_4w_change'].iloc[-1]:+,.0f}")
print(f"  Speculator 1W Change: {es_df['noncommercial_change'].iloc[-1]:+,.0f}")
print(f"  Speculator 4W Change: {es_df['noncommercial_4w_change'].iloc[-1]:+,.0f}")


In [ ]:
# Position change visualization
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=['Weekly Position Changes', '4-Week Cumulative Changes'],
    vertical_spacing=0.1
)

# Weekly
fig.add_trace(
    go.Bar(
        x=es_df.index[-52:], y=es_df['commercial_change'].iloc[-52:],
        name='Commercial', marker_color='blue'
    ),
    row=1, col=1
)
fig.add_trace(
    go.Bar(
        x=es_df.index[-52:], y=es_df['noncommercial_change'].iloc[-52:],
        name='Speculator', marker_color='orange'
    ),
    row=1, col=1
)

# 4-Week
fig.add_trace(
    go.Scatter(
        x=es_df.index[-52:], y=es_df['commercial_4w_change'].iloc[-52:],
        mode='lines', name='Commercial 4W',
        line=dict(color='blue', width=2)
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=es_df.index[-52:], y=es_df['noncommercial_4w_change'].iloc[-52:],
        mode='lines', name='Speculator 4W',
        line=dict(color='orange', width=2)
    ),
    row=2, col=1
)

fig.update_layout(
    title='E-Mini S&P 500 Positioning Changes (Last Year)',
    template='plotly_dark',
    height=550,
    barmode='group'
)
fig.show()


## 6. COT-Based Trading Strategy

In [ ]:
# Fetch SPY prices for backtesting
spy_prices = qj.eod.get_historical_prices(
    symbol='SPY',
    start_date='2020-01-01',
    end_date='2024-12-31',
    frequency='1d'
)
spy_data = spy_prices.get('value', spy_prices) if isinstance(spy_prices, dict) else spy_prices

if isinstance(spy_data, list) and len(spy_data) > 0:
    spy_df = pd.DataFrame(spy_data)
    spy_df['date'] = pd.to_datetime(spy_df['date'])
    spy_df = spy_df.set_index('date')
    spy_df['returns'] = spy_df['close'].pct_change()
    print(f"SPY prices: {len(spy_df)} days")
else:
    # Simulate prices
    print("Simulating SPY prices...")
    dates = pd.date_range(start='2020-01-01', end='2024-12-31', freq='B')
    np.random.seed(42)
    returns = np.random.randn(len(dates)) * 0.01 + 0.0003
    prices = 300 * np.cumprod(1 + returns)
    spy_df = pd.DataFrame({'close': prices, 'returns': returns}, index=dates)


In [ ]:
# Align COT with daily prices
# Forward fill COT data to daily frequency
es_daily = es_df[['commercial_zscore', 'noncommercial_zscore']].resample('D').ffill()

# Merge with SPY
merged = spy_df[['close', 'returns']].join(es_daily, how='left')
merged = merged.ffill().dropna()

# Strategy: Follow commercials (smart money) at extremes
# Long when commercial z-score > 1.5 (commercials bullish)
# Short when commercial z-score < -1.5 (commercials bearish)

merged['signal'] = 0
merged.loc[merged['commercial_zscore'] > 1.5, 'signal'] = 1
merged.loc[merged['commercial_zscore'] < -1.5, 'signal'] = -1

# Alternative: Fade speculators at extremes
merged['fade_signal'] = 0
merged.loc[merged['noncommercial_zscore'] > 2, 'fade_signal'] = -1  # Fade crowded longs
merged.loc[merged['noncommercial_zscore'] < -2, 'fade_signal'] = 1   # Fade crowded shorts

# Strategy returns
merged['cot_strategy'] = merged['signal'].shift(5) * merged['returns']  # 5-day lag for weekly COT
merged['fade_strategy'] = merged['fade_signal'].shift(5) * merged['returns']
merged['buy_hold'] = merged['returns']

# Cumulative returns
merged['cot_cumulative'] = (1 + merged['cot_strategy']).cumprod()
merged['fade_cumulative'] = (1 + merged['fade_strategy']).cumprod()
merged['bh_cumulative'] = (1 + merged['buy_hold']).cumprod()

print(f"\nBacktest Period: {merged.index.min().date()} to {merged.index.max().date()}")
print(f"Days with COT Long Signal: {(merged['signal'] == 1).sum()}")
print(f"Days with COT Short Signal: {(merged['signal'] == -1).sum()}")


In [ ]:
# Strategy performance comparison
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=merged.index, y=(merged['bh_cumulative'] - 1) * 100,
    mode='lines', name='Buy & Hold',
    line=dict(color='white', width=2)
))

fig.add_trace(go.Scatter(
    x=merged.index, y=(merged['cot_cumulative'] - 1) * 100,
    mode='lines', name='Follow Commercials',
    line=dict(color='cyan', width=2)
))

fig.add_trace(go.Scatter(
    x=merged.index, y=(merged['fade_cumulative'] - 1) * 100,
    mode='lines', name='Fade Speculators',
    line=dict(color='yellow', width=2)
))

fig.update_layout(
    title='COT Strategy Backtest vs Buy & Hold',
    xaxis_title='Date',
    yaxis_title='Cumulative Return (%)',
    template='plotly_dark',
    height=500
)
fig.show()


## 7. Strategy Performance Metrics

In [ ]:
def calculate_metrics(returns, name):
    """Calculate strategy performance metrics."""
    returns = returns.dropna()
    if len(returns) == 0:
        return None
    
    total_return = (1 + returns).prod() - 1
    years = len(returns) / 252
    ann_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    # Max drawdown
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min()
    
    # Win rate
    win_rate = (returns > 0).mean()
    
    return {
        'Strategy': name,
        'Total Return': total_return * 100,
        'Ann. Return': ann_return * 100,
        'Ann. Vol': ann_vol * 100,
        'Sharpe': sharpe,
        'Max DD': max_dd * 100,
        'Win Rate': win_rate * 100
    }

metrics = [
    calculate_metrics(merged['buy_hold'], 'Buy & Hold'),
    calculate_metrics(merged['cot_strategy'], 'Follow Commercials'),
    calculate_metrics(merged['fade_strategy'], 'Fade Speculators')
]

metrics_df = pd.DataFrame([m for m in metrics if m is not None])
print("\nStrategy Performance Comparison:")
print(metrics_df.round(2).to_string(index=False))


## 8. Commercial vs Speculator Sentiment

In [ ]:
# Sentiment divergence indicator
es_df['sentiment_divergence'] = es_df['commercial_zscore'] - es_df['noncommercial_zscore']

# Interpretation:
# Positive divergence = Commercials more bullish than specs (bullish signal)
# Negative divergence = Specs more bullish than commercials (bearish signal)

print("\nSentiment Divergence Analysis:")
print(f"  Current Divergence: {es_df['sentiment_divergence'].iloc[-1]:.2f}")
print(f"  Historical Mean: {es_df['sentiment_divergence'].mean():.2f}")
print(f"  Historical Std: {es_df['sentiment_divergence'].std():.2f}")


In [ ]:
# Divergence plot
fig = go.Figure()

colors = ['green' if x > 0 else 'red' for x in es_df['sentiment_divergence'].iloc[-104:]]

fig.add_trace(go.Bar(
    x=es_df.index[-104:],
    y=es_df['sentiment_divergence'].iloc[-104:],
    marker_color=colors,
    name='Divergence'
))

fig.add_hline(y=1, line_dash="dash", line_color="yellow", annotation_text="Bullish Zone")
fig.add_hline(y=-1, line_dash="dash", line_color="yellow", annotation_text="Bearish Zone")

fig.update_layout(
    title='Commercial vs Speculator Sentiment Divergence (2Y)',
    xaxis_title='Date',
    yaxis_title='Divergence (Commercial Z - Speculator Z)',
    template='plotly_dark',
    height=450
)
fig.show()


## 9. Summary Report

In [ ]:
print("="*70)
print("COT POSITIONING & SENTIMENT ANALYSIS")
print("="*70)

print(f"\n1. CURRENT POSITIONING (E-Mini S&P 500)")
print(f"   Commercial Net: {es_df['commercial_net'].iloc[-1]:+,.0f}")
print(f"   Commercial Z-Score: {es_df['commercial_zscore'].iloc[-1]:+.2f}")
print(f"   Speculator Net: {es_df['noncommercial_net'].iloc[-1]:+,.0f}")
print(f"   Speculator Z-Score: {es_df['noncommercial_zscore'].iloc[-1]:+.2f}")

print(f"\n2. SENTIMENT SIGNALS")
divergence = es_df['sentiment_divergence'].iloc[-1]
if divergence > 1:
    signal = "BULLISH - Commercials significantly more bullish"
elif divergence < -1:
    signal = "BEARISH - Speculators overly bullish"
else:
    signal = "NEUTRAL - No extreme divergence"
print(f"   Signal: {signal}")
print(f"   Divergence: {divergence:.2f}")

print(f"\n3. MULTI-ASSET EXTREMES")
for _, row in zscore_df.iterrows():
    if abs(row['Commercial Z']) > 1.5 or abs(row['Speculator Z']) > 1.5:
        print(f"   {row['Name']}: Comm Z={row['Commercial Z']:.1f}, Spec Z={row['Speculator Z']:.1f}")

print(f"\n4. STRATEGY BACKTEST")
if len(metrics_df) > 0:
    best = metrics_df.loc[metrics_df['Sharpe'].idxmax()]
    print(f"   Best Strategy: {best['Strategy']}")
    print(f"   Sharpe Ratio: {best['Sharpe']:.2f}")
    print(f"   Total Return: {best['Total Return']:.1f}%")

print(f"\n5. KEY INSIGHTS")
print(f"   - Commercial positioning often leads price moves")
print(f"   - Extreme speculator positioning = potential reversal")
print(f"   - COT signals work best at extremes (Z > ±1.5)")
print(f"   - Weekly data = medium-term timing, not day trading")

print("\n" + "="*70)


## Summary

This CQF-level example covered:

1. **COT Data Structure**: Commercial vs Non-Commercial positioning
2. **Z-Score Analysis**: Detecting extreme positioning
3. **Multi-Asset Dashboard**: Cross-market sentiment
4. **Position Flow**: Weekly and 4-week changes
5. **Trading Strategies**: Follow commercials, fade speculators
6. **Backtesting**: Strategy performance vs buy & hold
7. **Sentiment Divergence**: Commercial vs Speculator indicator

**Applications**:
- Macro regime identification
- Contrarian trading signals
- Risk overlay for portfolios
- Commodity trading strategies